##  Feature engineering
find the best feature enginerring techniques out of (Bag-of-words vs tfidf) and also which variant will be best uni-gram or bi-gram or tri-gram

In [1]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = userdata.get('AWS_DEFAULT_REGION')
os.environ["satya_mlflow_ec2_uri"] = userdata.get('satya_mlflow_ec2_uri')

In [2]:
!pip install mlflow boto3 awscli
!aws sts get-caller-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.0/314.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/8

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import mlflow


In [5]:
from google.colab import drive
drive.mount('/content/drive')

# loading the pre-processed data (prevent from preprocessing again and again)
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Youtube_Comment_Sentiment_Analysis/reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(36662, 2)

In [6]:
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri(os.environ["satya_mlflow_ec2_uri"])

# Set or create an experiment
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

2025/09/24 07:05:22 INFO mlflow.tracking.fluent: Experiment with name 'Exp 2 - BoW vs TfIdf' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://satya-mlflow-bucket/605021623909563050', creation_time=1758697532400, experiment_id='605021623909563050', last_update_time=1758697532400, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={}>

In [7]:
# we are trying two types of vectorizers (Bag-of-words vs tfidf)

# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train) # train_test_split before TF-IDF and BoW to ensure no data Leakage
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, name=f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)]  # unigrams, bigrams, trigrams
max_features = 5000  # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")


# Thus the above code will run 6 times

2025/09/24 07:12:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/24 07:15:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run BoW_(1, 1)_RandomForest at: http://65.2.37.109:5000/#/experiments/605021623909563050/runs/0aa187ba69204856a9617effcf994605
🧪 View experiment at: http://65.2.37.109:5000/#/experiments/605021623909563050


2025/09/24 07:21:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/24 07:21:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 1)_RandomForest at: http://65.2.37.109:5000/#/experiments/605021623909563050/runs/937b391b625542e99eb106681aa81c72
🧪 View experiment at: http://65.2.37.109:5000/#/experiments/605021623909563050


2025/09/24 07:22:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/24 07:23:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: http://65.2.37.109:5000/#/experiments/605021623909563050/runs/3b12fc4112414dcd9d9fa7afdbcd3511
🧪 View experiment at: http://65.2.37.109:5000/#/experiments/605021623909563050


2025/09/24 07:26:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/24 07:26:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run BoW_(1, 3)_RandomForest at: http://65.2.37.109:5000/#/experiments/605021623909563050/runs/3808d44c663b4f7989033b72c6b7339b
🧪 View experiment at: http://65.2.37.109:5000/#/experiments/605021623909563050


2025/09/24 07:28:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/24 07:28:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 3)_RandomForest at: http://65.2.37.109:5000/#/experiments/605021623909563050/runs/ec42aeeb3a7147f3949d206e9c23d8f0
🧪 View experiment at: http://65.2.37.109:5000/#/experiments/605021623909563050


Now, we have to go to the MLflow

and compare the combinations of Bag-of-words and tfidf with *unigrams, bigrams and trigrams*, for the parameter precision, recall and acccuracy.

select all the experiments then click on compare then select ngram_range and vectorizer_type as parameters and accuracy, -1_precision and -1_recall as metrics.

Hence, the conclusion is that **tfidf with the tri-gram** is giving the best result i.e. recall and accuracy is high but precision is high but not compared to other combination. But, our targets was recall and accuracy so continuing with tfidf with the tri-gram.

if you find other combination as best result then you can go with that like BoW with bi-gram or tfidf with the bi-gram.